# $(SASA) Models - Kmeans$

In [1]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pickle
import numpy as np
import pandas as pd

from pmbrl.model2 import Base_Line_Simple_Model
# from pmbrl.model2 import Model, Regularized_Reference_Loss
from pmbrl.data import Experiment_Data, get_data_expanded, get_data_compacted

In [2]:
nome_do_arquivo = 'kmodels.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    models = exp['model']

del exp
del arquivo

In [ ]:
data = Experiment_Data()

data.load(path='../testing_data.csv')

expansions = {
    's': ['s0', 's1', 's2', 's3'],
    's_': ['s_0', 's_1', 's_2', 's_3'],
    's__': ['s__0', 's__1', 's__2', 's__3'],
}

df = get_data_expanded(data.build_training_dataset(), expansions)
# df = df.loc[df['episode']<15].copy()
df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,s2,s3,s_0,s_1,s_2,s_3,s__0,s__1,s__2,s__3
758,9,38,"(0.1011387519285949, 1.9659375713404728)","(0.024, 0.16, 0.014, -0.047)",1,1.0,"(0.027, 0.345, 0.013, -0.12)",1.0,1.0,"(0.034, 0.53, 0.01, -0.194)",...,0.014,-0.047,0.027,0.345,0.013,-0.120,0.034,0.530,0.010,-0.194
1239,17,59,"(0.249678455417393, 1.716114786229628)","(0.087, 0.223, -0.091, -0.209)",0,1.0,"(0.092, 0.038, -0.095, -0.123)",1.0,1.0,"(0.093, 0.224, -0.097, -0.228)",...,-0.091,-0.209,0.092,0.038,-0.095,-0.123,0.093,0.224,-0.097,-0.228
1726,14,87,"(0.2009601517301062, 0.9751414354812274)","(-0.102, -0.349, 0.06, 0.305)",0,1.0,"(-0.109, -0.539, 0.066, 0.478)",0.0,1.0,"(-0.12, -0.729, 0.075, 0.651)",...,0.060,0.305,-0.109,-0.539,0.066,0.478,-0.120,-0.729,0.075,0.651
1217,8,58,"(0.2280266930676658, 0.5206197145652827)","(0.003, 0.434, -0.06, -0.738)",1,1.0,"(0.012, 0.631, -0.075, -1.068)",1.0,1.0,"(0.024, 0.827, -0.096, -1.402)",...,-0.060,-0.738,0.012,0.631,-0.075,-1.068,0.024,0.827,-0.096,-1.402
935,6,47,"(0.1677354969823425, 0.9471294840196408)","(0.016, -0.353, 0.001, 0.283)",0,1.0,"(0.009, -0.542, 0.007, 0.446)",0.0,1.0,"(-0.002, -0.731, 0.016, 0.61)",...,0.001,0.283,0.009,-0.542,0.007,0.446,-0.002,-0.731,0.016,0.610


# Predict 

In [4]:
def optim_params(prediction_dataset):
    weights = prediction_dataset.copy()
    for m, _ in enumerate(models):
        weights['estimated_s'] = prediction_dataset[f'estimated_s_model_{m}']
        expansions = {f'estimated_s': [f'estimated_s0', f'estimated_s1', f'estimated_s2', f'estimated_s3']}
        ds = get_data_expanded(weights, expansions)
        
        for d in range(4):
            weights[f'estimated_s{d}_model{m}'] = ds[f'estimated_s{d}']
    weights['step_A'] = weights.apply(lambda row: [[row[f'estimated_s{d}_model{m}'] for m,_ in enumerate(models)] for d in range(4)], axis=1)
    weights['step_b'] = weights.apply(lambda row: [row[f's__{d}'] for d in range(4)], axis=1)

    weights['concat_A'] = weights.apply(lambda row: np.array(weights.loc[(weights['episode']==row['episode'])&(weights['step']<=row['step'])].step_A.to_list()), axis=1)
    weights['concat_b'] = weights.apply(lambda row: np.array(weights.loc[(weights['episode']==row['episode'])&(weights['step']<=row['step'])].step_b.to_list()), axis=1)
    
    weights['A'] = weights.apply(lambda row: row.concat_A.reshape((row.concat_A.shape[0]*row.concat_A.shape[1], row.concat_A.shape[2])), axis=1)
    weights['b'] = weights.apply(lambda row: row.concat_b.reshape((row.concat_b.shape[0]*row.concat_b.shape[1],)), axis=1)
    weights['w'] = weights.apply(lambda row: np.linalg.lstsq(row.A, row.b)[0], axis=1)

    weights[[f'estimated_weight_{m}' for m, _ in enumerate(models)]] = weights.apply(lambda row: pd.Series(row['w']),axis=1)
    return weights


In [5]:

def predict_with_params(prediction_dataset, weights):
    for m, _ in enumerate(models):
        prediction_dataset[f'weight_{m}'] = weights[f'estimated_weight_{m}']

    expansions = {
        f'estimated_s_model_{m}': [f'estimated_s{d}_model{m}' for d in range(4)] for m,_ in enumerate(models)}
    ds = get_data_expanded(prediction_dataset, expansions)
    
    for d in range(4):
        ds[f'estimated_weighted_s{d}'] = ds.apply(
            lambda row: np.sum([row[f'estimated_s{d}_model{m}']* row[f'weight_{m}'] for m, _ in enumerate(models)])
            , axis=1
        )
    
    ds = get_data_compacted(ds, {'estimated_weighted_s': [f'estimated_weighted_s{d}' for d in range(4)]})
     
    prediction_dataset['estimated_r'] = ds['estimated_r_model_0']
    prediction_dataset['estimated_s'] = ds['estimated_weighted_s']
    
    results = data.get_evaluation_metrics(prediction_dataset, p=False)
    prediction_dataset[f'rse'] = results['rse']
    prediction_dataset[f'rse_normalized'] = results['rse_normalized']

    prediction_dataset[f'rse_s0'] = results['rse_s0']
    prediction_dataset[f'rse_s1'] = results['rse_s1']
    prediction_dataset[f'rse_s2'] = results['rse_s2']
    prediction_dataset[f'rse_s3'] = results['rse_s3']

    prediction_dataset[f'rse_s0_normalized'] = results['rse_s0_normalized']
    prediction_dataset[f'rse_s1_normalized'] = results['rse_s1_normalized']
    prediction_dataset[f'rse_s2_normalized'] = results['rse_s2_normalized']
    prediction_dataset[f'rse_s3_normalized'] = results['rse_s3_normalized']

    return prediction_dataset

In [6]:
def evaluate(models, df):
    def predict(model):
        prediction_dataset = df.copy()
        prediction_dataset[model.grouped_targets_lables] = prediction_dataset.apply(lambda row: data._predict_from_row(row, model), axis=1, result_type='expand')
        return prediction_dataset

    predictions = [predict(m) for m in models]

    prediction_dataset = df.copy()
    for i, pred in enumerate(predictions):
        prediction_dataset[f'estimated_r_model_{i}'] = pred['estimated_r']
        prediction_dataset[f'estimated_s_model_{i}'] = pred['estimated_s']
        
        results = data.get_evaluation_metrics(pred, p=False)
        prediction_dataset[f'rse_model_{i}'] = results['rse']
        prediction_dataset[f'rse_normalized_model_{i}'] = results['rse_normalized']

        prediction_dataset[f'rse_s0_model_{i}'] = results['rse_s0']
        prediction_dataset[f'rse_s1_model_{i}'] = results['rse_s1']
        prediction_dataset[f'rse_s2_model_{i}'] = results['rse_s2']
        prediction_dataset[f'rse_s3_model_{i}'] = results['rse_s3']

        prediction_dataset[f'rse_s0_normalized_model_{i}'] = results['rse_s0_normalized']
        prediction_dataset[f'rse_s1_normalized_model_{i}'] = results['rse_s1_normalized']
        prediction_dataset[f'rse_s2_normalized_model_{i}'] = results['rse_s2_normalized']
        prediction_dataset[f'rse_s3_normalized_model_{i}'] = results['rse_s3_normalized']


    return prediction_dataset

In [7]:
def predicts(models, df):
    pre_df = df.copy()
    pre_df[['s_', 's_0', 's_1', 's_2', 's_3', 'a_', 's__0', 's__1', 's__2', 's__3', 's__']] = pre_df[['s', 's0', 's1', 's2', 's3', 'a', 's_0', 's_1', 's_2', 's_3', 's_']]

    prediction_dataset = evaluate(models, pre_df)
    params = optim_params(prediction_dataset)
    prediction_dataset = evaluate(models, df)
    final_predictions = predict_with_params(prediction_dataset, params)

    cols = [
        's__', 'estimated_s', 'rse', 'rse_normalized',
        'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3', 
        'rse_s0_normalized', 'rse_s1_normalized',
        'rse_s2_normalized', 'rse_s3_normalized'
    ] + [f'weight_{m}' for m, _ in enumerate(models)] + [f'estimated_s_model_{m}' for m, _ in enumerate(models)]

    return final_predictions[cols]

In [8]:
prediction_dataset = predicts(models, df)
prediction_dataset.head()

,s__,estimated_s,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,...,weight_0,weight_1,weight_2,weight_3,weight_4,estimated_s_model_0,estimated_s_model_1,estimated_s_model_2,estimated_s_model_3,estimated_s_model_4
758,"(0.034, 0.53, 0.01, -0.194)","(0.03720320889189741, 0.5284197723497355, 0.00...",0.008240,0.503088,0.003203,0.001580,0.000184,0.003272,0.549317,0.526445,...,-0.358112,0.004754,0.019976,0.007600,1.307686,"(0.023, 0.536, 0.01, -0.475)","(0.027, 0.451, 0.012, 0.897)","(0.03, 0.523, 0.009, -0.44)","(0.033, 0.558, 0.011, -0.77)","(0.034, 0.538, 0.01, -0.273)"
1239,"(0.093, 0.224, -0.097, -0.228)","(0.09468180048726493, 0.22724626519129626, -0....",0.012958,0.504218,0.001682,0.003246,0.002486,0.005545,0.547710,0.526855,...,-0.164724,0.002052,-0.174553,0.020643,1.306113,"(0.092, 0.24, -0.098, -0.577)","(0.074, 0.236, -0.114, -0.247)","(0.093, 0.239, -0.099, -0.484)","(0.089, 0.242, -0.1, -0.688)","(0.095, 0.232, -0.1, -0.305)"
1726,"(-0.12, -0.729, 0.075, 0.651)","(-0.11990104851962621, -0.7208980570535498, 0....",0.015306,0.502777,0.000099,0.008102,0.000235,0.006870,0.546039,0.528049,...,0.117530,-0.000424,-0.413445,0.034148,1.256048,"(-0.118, -0.747, 0.078, 0.969)","(-0.37, -0.74, 0.036, 1.676)","(-0.121, -0.743, 0.077, 0.82)","(-0.124, -0.657, 0.072, 0.867)","(-0.121, -0.731, 0.076, 0.68)"
1217,"(0.024, 0.827, -0.096, -1.402)","(0.024514518750266827, 0.8317560422935137, -0....",0.013292,0.503604,0.000515,0.004756,0.001797,0.006224,0.546478,0.527226,...,0.110911,-0.001070,0.779255,-0.004243,0.121264,"(0.022, 0.83, -0.099, -1.505)","(0.015, 0.763, -0.07, -1.454)","(0.026, 0.824, -0.096, -1.394)","(0.026, 0.842, -0.096, -1.68)","(0.016, 0.841, -0.103, -1.35)"
935,"(-0.002, -0.731, 0.016, 0.61)","(-0.0029170580762917257, -0.7283688800029421, ...",0.012574,0.503137,0.000917,0.002631,0.000994,0.008032,0.546903,0.526704,...,-0.242433,0.007633,0.378309,-0.029603,0.871327,"(0.0, -0.743, 0.017, 0.887)","(-0.078, -0.729, 0.002, 1.002)","(-0.002, -0.753, 0.015, 0.793)","(-0.006, -0.667, 0.017, 0.912)","(-0.002, -0.732, 0.016, 0.634)"


In [9]:
prediction_dataset.describe()

,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized,weight_0,weight_1,weight_2,weight_3,weight_4
count,2127.000000,2127.000000,2.127000e+03,2127.000000,2.127000e+03,2.127000e+03,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000
mean,0.070166,0.508544,7.859759e-03,0.011514,4.686200e-03,4.610572e-02,0.554234,0.528887,0.512437,0.438616,0.236758,0.028362,0.274691,0.075905,0.379809
std,0.523130,0.042889,7.134187e-02,0.072457,2.397384e-02,3.760335e-01,0.075335,0.017811,0.057491,0.032175,0.969614,0.208467,0.648049,0.529789,0.614151
min,0.001112,0.502070,1.063791e-08,0.000002,2.046881e-07,3.101859e-07,0.545935,0.526057,0.501200,0.434671,-22.459615,-2.003953,-9.338019,-10.606972,-3.800425
25%,0.009589,0.503133,7.183558e-04,0.001506,6.534369e-04,3.536055e-03,0.546693,0.526427,0.502766,0.434974,-0.130957,-0.005846,-0.016605,-0.050799,-0.016252
50%,0.017234,0.503991,1.690692e-03,0.003450,1.532501e-03,9.097164e-03,0.547720,0.526905,0.504874,0.435449,0.060605,-0.001471,0.254312,0.009347,0.398639
75%,0.037682,0.505909,3.881927e-03,0.007754,3.142913e-03,2.200805e-02,0.550034,0.527963,0.508736,0.436554,0.454569,0.004492,0.615390,0.095491,0.843498
max,20.748240,2.111326,2.554675e+00,2.549124,7.673646e-01,1.487708e+01,3.243585,1.152685,2.341402,1.707630,19.450473,4.388366,10.763391,10.190670,2.336506
